# Очистка данных: Spend

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

# Пути к данным
RAW_DATA_DIR    = os.path.join('..', 'Sources')
CLEANED_DIR     = os.path.join('..', 'data', 'cleaned')

SPEND_INPUT  = os.path.join(RAW_DATA_DIR, 'Spend (Done).xlsx')
SPEND_OUTPUT = os.path.join(CLEANED_DIR, 'spend_clean.pkl')

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

## Загрузка и первичный осмотр

In [2]:
df = pd.read_excel(SPEND_INPUT)

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

print(f'Форма: {df.shape}')

n_before = df.shape[0]

h.descr_df(df, include='all', show_sample_rows=False)

Форма: (20779, 8)


,Тип,Заполнено,Пропуски,% Пропусков,Уникальных,Min,Mean,Median,Max,Range
Признак,,,,,,,,,,
date,datetime64[us],20779,0,0.00,355,NaN,NaN,NaN,NaN,NaN
source,str,20779,0,0.00,14,NaN,NaN,NaN,NaN,NaN
campaign,str,14785,5994,28.85,51,NaN,NaN,NaN,NaN,NaN
impressions,int64,20779,0,0.00,4003,0.00,2458.20,63.00,431445.00,431445.00
spend,float64,20779,0,0.00,2859,0.00,7.20,0.58,774.00,774.00
clicks,int64,20779,0,0.00,552,0.00,23.99,1.00,2415.00,2415.00
adgroup,str,13951,6828,32.86,24,NaN,NaN,NaN,NaN,NaN
ad,str,13951,6828,32.86,176,NaN,NaN,NaN,NaN,NaN


In [6]:
# У Spend нет Id — дубликат это полное совпадение всех полей
full_dupes = df.duplicated().sum()
print(f'Полных дубликатов: {full_dupes}')

Полных дубликатов: 917


In [7]:
if full_dupes > 0:
    print("Итоги по числовым полям в дубликатах (сумма):")
    # Вычисляем суммы только для строк, которые будут удалены (keep='first' оставит одну, остальные в расчет)
    removed_dupes = df[df.duplicated(keep='first')]
    summary_dupes = removed_dupes[['impressions', 'spend', 'clicks']].sum()
    display(summary_dupes.to_frame('Сумма удаляемых данных'))
    
    df = df.drop_duplicates().reset_index(drop=True)

print(f'Строк до: {n_before}  →  после: {len(df)}  (удалено: {n_before - len(df)})')

Итоги по числовым полям в дубликатах (сумма):


,Сумма удаляемых данных
impressions,0.00
spend,0.00
clicks,46.00


Строк до: 20779  →  после: 19862  (удалено: 917)


## Типы данных: дата

In [9]:
DATE_COLS = ['date']

for col in DATE_COLS:
    # Явно указываем формат YYYY-MM-DD
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT = не распарсились
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')

print(f'Диапазон дат: {df[DATE_COLS[0]].min()}  →  {df[DATE_COLS[0]].max()}')

Типы после парсинга:
date    datetime64[us]
dtype: object

date: NaT = 0 (0.00%)
Диапазон дат: 2023-07-03 00:00:00  →  2024-06-21 00:00:00


## Числовые поля

In [10]:
# Проверяем наличие "грязных" значений в числовых полях
NUM_COLS = ['impressions', 'spend', 'clicks']

for col in NUM_COLS:
    print(f'--- {col} ---')
    print(f'  Тип: {df[col].dtype}')
    print(f'  Мин: {df[col].min()}, Макс: {df[col].max()}')
    neg = (df[col] < 0).sum()
    print(f'  Отрицательных значений: {neg}')

--- impressions ---
  Тип: int64
  Мин: 0, Макс: 431445
  Отрицательных значений: 0
--- spend ---
  Тип: float64
  Мин: 0.0, Макс: 774.0
  Отрицательных значений: 0
--- clicks ---
  Тип: int64
  Мин: 0, Макс: 2415
  Отрицательных значений: 0


## Пропущенные значения

Этот датасет это единственный источник значений campaign, adgroup, ad, поэтому дозаполнить их не 
получиться. Пропущенные значения заменяем на Unknown и меняем тип на category

In [11]:
# campaign, adgroup, ad — ~30% пропусков, заполняем 'Unknown'
FILL_UNKNOWN = ['campaign', 'adgroup', 'ad']

for col in FILL_UNKNOWN:
    n_miss = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

# Проверка
print('\nПропуски после заполнения:')
missing_after = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': missing_after, '% пропусков': (missing_after / len(df) * 100).round(2)})
    .query('Пропуски > 0')
)

CAT_COLS = ['source', 'campaign', 'adgroup', 'ad']

for col in CAT_COLS:
    # Нормализация: убираем лишние пробелы по краям
    df[col] = df[col].str.strip()
    df[col] = df[col].astype('category')
    print(f'{col}: {df[col].nunique()} уникальных значений')

campaign: заполнено 5077 пропусков → "Unknown"
adgroup: заполнено 5911 пропусков → "Unknown"
ad: заполнено 5911 пропусков → "Unknown"

Пропуски после заполнения:


,Пропуски,% пропусков


source: 14 уникальных значений
campaign: 52 уникальных значений
adgroup: 25 уникальных значений
ad: 177 уникальных значений


## Итоговый осмотр

In [12]:
h.descr_df(df, include='all', show_stats=True, show_sample_rows=True, show_quartiles=True)

,Тип,Заполнено,Пропуски,% Пропусков,Уникальных,Пример 1,Пример 2,Пример 3,Min,Mean,Median,Max,Range,Q1,Q3,IQR
Признак,,,,,,,,,,,,,,,,
date,datetime64[us],19862,0,0.00,355,2023-07-03 00:00:00,2023-07-03 00:00:00,2023-07-03 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,category,19862,0,0.00,14,Google Ads,Google Ads,Facebook Ads,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign,category,19862,0,0.00,52,gen_analyst_DE,performancemax_eng_DE,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
impressions,int64,19862,0,0.00,4003,6,4,0,0.00,2571.70,82.00,431445.00,431445.00,1.00,760.75,759.75
spend,float64,19862,0,0.00,2859,0.00,0.01,0.00,0.00,7.53,0.74,774.00,774.00,0.00,6.16,6.16
clicks,int64,19862,0,0.00,552,0,1,0,0.00,25.10,2.00,2415.00,2415.00,0.00,13.00,13.00
adgroup,category,19862,0,0.00,25,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ad,category,19862,0,0.00,177,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Сохранение

In [13]:
df.to_pickle(SPEND_OUTPUT)
df.to_excel(SPEND_OUTPUT.replace('.pkl', '.xlsx'))

# Итоги
summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Диапазон дат',
        'Каналов (source)',
        'Итого Spend, €',
        'Пропуски после заполнения'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        f'{df["date"].min().date()} → {df["date"].max().date()}',
        df['source'].nunique(),
        f'{df["spend"].sum():,.2f}',
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {SPEND_OUTPUT}')
display(pd.DataFrame(summary_data))


Сохранено: ../data/cleaned/spend_clean.pkl


,Метрика,Значение
0,Строк исходно,20779
1,Строк после очистки,19862
2,Удалено дубликатов,917
3,Диапазон дат,2023-07-03 → 2024-06-21
4,Каналов (source),14
5,"Итого Spend, €","149,523.45"
6,Пропуски после заполнения,0


## Описание датасета

**Источник:** `Spend.xlsx` — данные рекламных кабинетов  
**Назначение:** учет маркетинговых затрат для расчета ROI/ROAS и анализа эффективности каналов привлечения

| Столбец | Тип | Описание |
|---|---|---|
| `date` | `datetime` | Дата расхода |
| `source` | `category` | Рекламный канал/источник (Google, Facebook и др.) |
| `campaign` | `category` | Название рекламной кампании |
| `impressions` | `int64` | Количество показов объявлений |
| `spend` | `float64` | Сумма фактических затрат в валюте кабинета |
| `clicks` | `int64` | Количество переходов (кликов) |
| `adgroup` | `category` | Группа объявлений |
| `ad` | `category` | Конкретное объявление/креатив |

**Объём:** 19 862 записей (после очистки), 8 столбцов  

**Особенности анализа:**
- **Фильтрация по порогу Spend >= 1 €:** для корректной оценки "живой" рекламы в блоке статистики выведены дополнительные метрики, исключающие микро-траты и технические выгрузки без реальной активности. Это позволяет увидеть реальный средний чек и объем данных, влияющих на бюджет.
- **Ключевые связи:** `date` + `source` + `campaign` → агрегация для сопоставления с результатами продаж в `06_analytics`.

## Выводы

В данных рекламных кабинетов выявлено **917 полных дублей (~4,4%)** и **~30% незаполненных значений** в полях `campaign`, `adgroup`, `ad`. Дубли возникают при повторной выгрузке одного и того же периода из рекламного кабинета — классическая проблема ручных экспортов. 

Если не удалить дубли, суммарные расходы будут задвоены или утроены по отдельным периодам — ROMI окажется заниженным, а бюджет по каналам не сойдётся с реальными тратами. Пропущенные кампании без обработки либо выпадут при объединении с данными сделок, потеряв часть фактических расходов, либо сломают агрегацию по источникам.

**Что сделано:** полные дубли удалены без потери финансовых данных (каждый дубль был 100%-копией исходной записи). Пропуски в `campaign`, `adgroup`, `ad` заполнены значением `'Unknown'` — это позволяет включить эти записи в агрегированную статистику расходов, а не терять их при группировке.

> **Системная рекомендация:** автоматизировать выгрузку через API (Google Ads / Meta Ads) с дедупликацией по `(date, source, campaign, ad)` на уровне ETL, а не вручную.